# 🕌 FajrGuard -- Wudu Detector Dataset Generator

This notebook automates the full pipeline to create **paired dry/wet face images** from CelebA,
which are then used to train `wudu_detector.tflite` -- the on-device MobileNetV2 classifier
that powers wudu verification in the FajrGuard app.

## Pipeline
```
CelebA (dry faces)
    → Face align & crop (MTCNN)
    → Stable Diffusion img2img + ControlNet (wet face synthesis)
    → Quality filter (blur + face detection)
    → Save paired output: {id}_dry.jpg + {id}_wet.jpg
    → metadata.csv
    → (Next step) Train MobileNetV2 → export wudu_detector.tflite
```

**Output feeds into:** `ml/train/train_wudu_model.py` → `ml/export/export_tflite.py` → `mobile/assets/models/wudu_detector.tflite`

## ⚙️ Cell 1 -- Runtime Check & Install Dependencies

In [ ]:
# Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    cc = torch.cuda.get_device_capability(0)
    print(f"Compute Capability: {cc[0]}.{cc[1]}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    if cc[0] < 7:
        print("GPU CC < 7.0 -- some CUDA kernels unavailable.")
        print("Wetness pipeline runs on CPU and works fine.")
else:
    print("No GPU. Wetness pipeline runs on CPU -- still works.")



In [ ]:
import torch
print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    cc = torch.cuda.get_device_capability(0)
    print(f"Compute Capability: {cc[0]}.{cc[1]}")
else:
    print("No GPU detected. Pipeline runs on CPU -- still works.")

# Core packages for wetness pipeline (pure CPU OpenCV/PIL/Numpy)
# No GPU needed for procedural wetness generation
!pip install opencv-python-headless facenet-pytorch Pillow tqdm imagehash kagglehub -q

print("
All dependencies ready")
print("Wetness generation: 100% CPU post-processing (OpenCV + PIL + Numpy)")



## 📁 Cell 2 -- Mount Drive & Configure Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ─── CONFIGURE THESE PATHS ───────────────────────────────────────────────────
CELEBA_DIR   = "/content/drive/MyDrive/celeba/img_align_celeba"  # overridden by kagglehub cell below if used
OUTPUT_DIR   = "/content/drive/MyDrive/fajrguard_ml/dataset"     # where pairs are saved
MAX_IMAGES   = 500        # start with 500; scale to 5000+ once pipeline is verified
STRENGTH     = 0.70       # img2img strength -- heavy wetness (was 0.52)
STEPS        = 40         # inference steps -- more detail for water droplets (was 28)
WET_THRESHOLD = 0.82      # matches WUDU_THRESHOLD in useWuduDetector.ts
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(f"{OUTPUT_DIR}/dry", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/wet", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/rejected", exist_ok=True)

print(f"✅ Output directory ready: {OUTPUT_DIR}")
print(f"📂 CelebA source: {CELEBA_DIR}")

if not os.path.exists(CELEBA_DIR):
    print("⚠️  CelebA directory not found! Upload CelebA to Google Drive first.")
    print("    Download from: https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html")
else:
    files = os.listdir(CELEBA_DIR)
    print(f"✅ Found {len(files):,} images in CelebA directory")


In [ ]:
# ─── Download CelebA via kagglehub (no manual Drive upload needed) ───────────
import kagglehub, os, glob

# First run: will prompt for kaggle.json credentials
# Upload kaggle.json when prompted, or run the cell below first:
#   from google.colab import files; files.upload()
#   !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

print('Downloading CelebA (~1.3 GB, cached after first run)...')
dl_path = kagglehub.dataset_download('zuozhaorui/celeba')
print(f'Downloaded to: {dl_path}')

# Auto-locate img_align_celeba folder
matches = glob.glob(f'{dl_path}/**/img_align_celeba', recursive=True)
if matches:
    CELEBA_DIR = matches[0]
else:
    CELEBA_DIR = dl_path  # fallback: images at root

print(f'✅ CELEBA_DIR = {CELEBA_DIR}')
print(f'   Images found: {len(os.listdir(CELEBA_DIR)):,}')

## 🌊 Cell 2b -- Import Real Wet Faces (User-Provided)
Place your real wet face images in a folder called `real_wet_faces/`
before running this cell. These will be mixed into training alongside
the synthetic wet faces generated from CelebA.

**Folder structure expected:**
```
real_wet_faces/
  ├── wet_face_001.jpg
  ├── wet_face_002.jpg
  └── ...
```

Each image should be a real photograph of a person with a wet face
(after wudu / washing / rain / shower). The face must be clearly visible.


In [ ]:
# Import real wet faces from Kaggle dataset -- used for BOTH training and holdout benchmarking
# Source: https://www.kaggle.com/datasets/abdulsamadmuyideen/real-wet-faces
import os, shutil, hashlib, glob, csv, random
from PIL import Image
from facenet_pytorch import MTCNN as _MTCNN

TRAINING_WET_DIR = f"{OUTPUT_DIR}/wet"
BENCHMARK_DIR    = f"{OUTPUT_DIR}/benchmark_wet"
os.makedirs(TRAINING_WET_DIR, exist_ok=True)
os.makedirs(BENCHMARK_DIR,    exist_ok=True)

REAL_WET_SOURCE = None

# Download via kagglehub (Colab)
try:
    import kagglehub
    print("Downloading real-wet-faces from Kaggle...")
    dl_path = kagglehub.dataset_download("abdulsamadmuyideen/real-wet-faces")
    for sub in sorted(os.listdir(dl_path)):
        sub_path = os.path.join(dl_path, sub)
        if os.path.isdir(sub_path):
            imgs = glob.glob(f"{sub_path}/*.jpg") + glob.glob(f"{sub_path}/*.png")
            if len(imgs) > 0:
                REAL_WET_SOURCE = sub_path
                break
    if REAL_WET_SOURCE is None:
        REAL_WET_SOURCE = dl_path
except Exception as e:
    print(f"kagglehub download failed: {e}")

if REAL_WET_SOURCE is None or not os.path.exists(REAL_WET_SOURCE):
    print("Real wet faces dataset not found.")
    print("   Make sure kaggle.json is configured or dataset is manually uploaded.")
else:
    print(f"Real wet faces source: {REAL_WET_SOURCE}")
    _det = _MTCNN(keep_all=False, device="cpu", min_face_size=60)
    kept, skipped_nf, skipped_dup = 0, 0, 0
    seen = set()
    exts = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
    files = sorted([f for f in os.listdir(REAL_WET_SOURCE)
                    if os.path.splitext(f)[1].lower() in exts])

    # Filter, deduplicate & collect valid face images
    valid_imgs = []
    for fname in files:
        fpath = os.path.join(REAL_WET_SOURCE, fname)
        try:
            img = Image.open(fpath).convert("RGB")
            if min(img.size) < 200:
                continue
            h = hashlib.md5(img.resize((64,64)).tobytes()).hexdigest()
            if h in seen:
                skipped_dup += 1; continue
            seen.add(h)
            boxes, probs = _det.detect(img)
            if boxes is None or probs is None:
                skipped_nf += 1; continue
            if not any(p is not None and float(p) > 0.85 for p in probs):
                skipped_nf += 1; continue
            valid_imgs.append(img)
        except Exception:
            pass

    # Split: 85% for training, 15% held out for final benchmark
    random.shuffle(valid_imgs)
    split_idx = int(len(valid_imgs) * 0.85)
    train_imgs   = valid_imgs[:split_idx]
    holdout_imgs = valid_imgs[split_idx:]

    # Save training real wet faces -> OUTPUT_DIR/wet/
    for i, img in enumerate(train_imgs):
        fname = f"real_wet_{i:05d}.jpg"
        dest = os.path.join(TRAINING_WET_DIR, fname)
        img.resize((512, 512), Image.LANCZOS).save(dest, quality=93)
        kept += 1

    # Save holdout benchmark images -> OUTPUT_DIR/benchmark_wet/ (NOT in training)
    for i, img in enumerate(holdout_imgs):
        dest = os.path.join(BENCHMARK_DIR, f"real_wet_holdout_{i:05d}.jpg")
        img.resize((512, 512), Image.LANCZOS).save(dest, quality=93)

    # Register training real wet faces in metadata.csv
    csv_path = f"{OUTPUT_DIR}/metadata.csv"
    csv_exists = os.path.exists(csv_path)
    with open(csv_path, "a", newline="") as csvfile:
        writer = csv.writer(csvfile)
        if not csv_exists:
            writer.writerow(["id", "source_file", "dry_path", "wet_path",
                             "wetness_score", "blur_score", "status", "reason", "timestamp"])
        for i in range(len(train_imgs)):
            wet_path = f"{OUTPUT_DIR}/wet/real_wet_{i:05d}.jpg"
            writer.writerow([f"real_wet_{i:05d}", f"real_wet_{i:05d}.jpg",
                             "", wet_path, "", "", "ok", "real_wet_benchmark", ""])

    print(f"Imported: {kept} real wet faces")
    print(f"  -> {len(train_imgs)} added to training set (OUTPUT_DIR/wet/ + metadata.csv)")
    print(f"  -> {len(holdout_imgs)} reserved for holdout benchmark (benchmark_wet/, excluded from training)")
    print(f"  Skipped: {skipped_nf} no-face, {skipped_dup} duplicates")


## 🤖 Cell 3 -- Load Models (SD + ControlNet + Face Detector)

In [ ]:
import torch
from facenet_pytorch import MTCNN

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

# Note: SD + ControlNet are NOT needed for wetness generation in Cell 4.
# The procedural wetness pipeline uses only OpenCV/PIL (no neural network).
# We load SD+ControlNet here only if you want to experiment with img2img.
# For the pure post-processing pipeline, only MTCNN is required.

print("Loading MTCNN (face detector)...")
mtcnn = MTCNN(keep_all=False, device=DEVICE, min_face_size=60)

print("\n✅ All models loaded")
print("   MTCNN             : face detection for mask creation")
print("   Wetness generation : pure OpenCV/PIL post-processing (Cell 4)")
print("   No SD / ControlNet needed for the default pipeline")
if DEVICE == "cuda":
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   GPU VRAM: {used:.1f} / {total:.1f} GB")


## 💧 Cell 4 -- Wet Face Generation Function

In [ ]:
from PIL import Image
import numpy as np
import cv2
import random

# ═══════════════════════════════════════════════════════════════════════════
# PHYSICS-BASED PROCEDURAL WETNESS -- 100% IDENTITY PRESERVATION
# Original face pixels are composited with water-effect layers using
# screen, overlay, and soft-light blending. No neural network touches
# the original pixels -- water effects are purely additive/post-process.
# ═══════════════════════════════════════════════════════════════════════════

# ═══ UNIFIED BLEND FUNCTIONS ═══════════════════════════════════════════════
def _blend(base, overlay, alpha, mode='screen'):
    '''Single unified blend: base + overlay with alpha, using any mode.'''
    b = base / 255.0
    o = overlay / 255.0
    a = alpha[:,:,None] if alpha.ndim == 2 else alpha
    if mode == 'screen':
        blended = 1.0 - (1.0 - b) * (1.0 - o)
    elif mode == 'overlay':
        lo = b < 0.5
        blended = np.where(lo, 2.0*b*o, 1.0 - 2.0*(1.0-b)*(1.0-o))
    elif mode == 'soft_light':
        lo = o < 0.5
        blended = np.where(lo, b - (1.0-2.0*o)*b*(1.0-b), b + (2.0*o-1.0)*(np.sqrt(b)-b))
    elif mode == 'linear_dodge':
        blended = np.clip(b + o, 0, 1)
    else:
        blended = 1.0 - (1.0 - b) * (1.0 - o)  # fallback to screen
    result = b*(1-a) + blended*a
    return np.clip(result*255, 0, 255)

# ═══ FACE MASK ═════════════════════════════════════════════════════════════
def create_face_mask(img_pil, detector, padding=0.22):
    '''Returns float32 (H,W) mask with soft elliptical face region.'''
    boxes, probs = detector.detect(img_pil)
    w, h = img_pil.size
    mask = np.zeros((h, w), dtype=np.float32)
    if boxes is not None and len(boxes) > 0:
        idx = int(probs.argmax())
        x1, y1, x2, y2 = boxes[idx]
        fw, fh = float(x2 - x1), float(y2 - y1)
        cx, cy = int((x1 + x2) / 2), int((y1 + y2) / 2)
        rx, ry = int(fw/2 * (1+padding)), int(fh/2 * (1+padding*1.3))
        cv2.ellipse(mask, (cx, cy), (rx, ry), 0, 0, 360, 1.0, -1)
        ksize = max(3, int(fw * 0.15)) | 1
        mask = cv2.GaussianBlur(mask, (ksize, ksize), fw * 0.08)
    else:
        cx, cy = w//2, h//2
        cv2.ellipse(mask, (cx, cy), (w//3, h//3), 0, 0, 360, 1.0, -1)
        mask = cv2.GaussianBlur(mask, (31, 31), 12)
    return np.clip(mask, 0, 1)

def _face_detail_mask(mask, grad_scale=8.0):
    '''Gradient magnitude of mask -- highlights edges/contours of face.'''
    gy, gx = np.gradient(mask)
    grad = np.sqrt(gx**2 + gy**2)
    grad = cv2.GaussianBlur(grad, (21, 21), 8)
    return np.clip(grad * grad_scale, 0, 1)

# ═══ WATER DROPLET TEMPLATES (PHYSICS-BASED) ═══════════════════════════════
def _make_drop_template(radius):
    '''Multi-layered water droplet: specular caustic + fresnel + refraction.'''
    sz = int(radius * 2 + 8)
    cy = cx = sz // 2
    Y, X = np.mgrid[0:sz, 0:sz]
    dx = (X - cx).astype(np.float32)
    dy = (Y - cy).astype(np.float32)
    dist = np.sqrt(dx*dx + dy*dy)
    nd = np.clip(dist / max(radius, 1), 0, 1.0)
    inside = (dist <= radius).astype(np.float32)
    # Alpha: soft falloff at edges (contact angle ≈ 70° for water on skin)
    alpha = np.clip(1.0 - nd**2.2, 0, 1) * 0.5 * inside
    # Specular caustic: bright crescent offset toward light source (top-left)
    sx = dx + radius * 0.30
    sy = dy + radius * 0.30
    sd = np.sqrt(sx*sx + sy*sy) / max(radius, 1)
    spec = np.clip(1.0 - sd * 1.8, 0, 1) ** 3.0  # sharper falloff for crisp highlight
    # Secondary bounce highlight (softer, opposite side)
    sx2 = dx - radius * 0.40
    sy2 = dy - radius * 0.40
    sd2 = np.sqrt(sx2*sx2 + sy2*sy2) / max(radius, 1)
    bounce = np.clip(1.0 - sd2 * 2.5, 0, 1) ** 4.0 * 0.15
    # Fresnel rim darkening (internal reflection)
    rim = nd ** 3.0 * 0.22 * inside
    # Combine: bright caustic + dim bounce - rim darkening
    bright = np.clip(0.05 + spec * 0.95 + bounce - rim, 0, 1) * inside
    return np.stack([bright, alpha], axis=-1)

_DROP_CACHE = {r: _make_drop_template(r) for r in range(1, 16)}

def _scatter_droplets(h, w, mask, strength=1.0):
    '''Cluster-aware droplet placement: droplets gather near each other.'''
    hl = np.zeros((h, w), dtype=np.float32)
    al = np.zeros((h, w), dtype=np.float32)
    ys, xs = np.where(mask > 0.30)
    if len(ys) < 10:
        return hl, al
    n = int(len(ys) * 0.055 * strength)
    # Weight small drops more, but allow more large drops for realism
    weights = [40, 30, 20, 12, 7, 4, 3, 2, 1, 1, 1, 1, 1, 1]
    # Seed several cluster centers -- droplets gravitate toward these
    n_clusters = max(3, n // 40)
    cluster_yx = [(int(ys[random.randint(0, len(ys)-1)]),
                   int(xs[random.randint(0, len(xs)-1)]))
                  for _ in range(n_clusters)]
    for _ in range(n):
        # 60% of drops near cluster centers, 40% randomly on face
        if random.random() < 0.60 and cluster_yx:
            cy_c, cx_c = random.choice(cluster_yx)
            cy = cy_c + random.randint(-80, 80)
            cx = cx_c + random.randint(-80, 80)
            cy = max(0, min(h-1, cy))
            cx = max(0, min(w-1, cx))
            if mask[cy, cx] < 0.15:
                idx = random.randint(0, len(ys)-1)
                cy, cx = int(ys[idx]), int(xs[idx])
        else:
            idx = random.randint(0, len(ys)-1)
            cy, cx = int(ys[idx]), int(xs[idx])
        r = random.choices(range(1, 15), weights=weights)[0]
        tmpl = _DROP_CACHE[r]
        tsz = tmpl.shape[0]
        y1, y2 = max(0, cy-tsz//2), min(h, cy-tsz//2+tsz)
        x1, x2 = max(0, cx-tsz//2), min(w, cx-tsz//2+tsz)
        dy1 = y1 - (cy - tsz//2)
        dx1 = x1 - (cx - tsz//2)
        patch = tmpl[dy1:dy1+(y2-y1), dx1:dx1+(x2-x1)]
        hl[y1:y2, x1:x2] = np.maximum(hl[y1:y2, x1:x2], patch[:,:,0])
        al[y1:y2, x1:x2] = np.maximum(al[y1:y2, x1:x2], patch[:,:,1])
    return hl, al

# ═══ MICRO-TEXTURE (WATER IN SKIN PORES) ══════════════════════════════════
def _micro_texture(h, w, mask, strength=1.0):
    '''Fine water-beading texture: simulates water filling skin micro-crevices.'''
    # Multi-octave noise for organic fine texture
    noise = np.zeros((h, w), dtype=np.float32)
    for octave, scale in enumerate([16, 32, 64]):
        nh, nw = h//scale, w//scale
        n = np.random.rand(nh, nw).astype(np.float32)
        n = cv2.resize(n, (w, h), interpolation=cv2.INTER_CUBIC)
        amp = 0.5 ** octave
        noise += n * amp
    noise = (noise - noise.mean()) / (noise.std() + 1e-6)
    noise = np.clip(noise * 0.5 + 0.5, 0, 1)
    # Sharpen the noise for beading effect (small bright spots)
    kernel = np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]], dtype=np.float32)/5.0
    noise = cv2.filter2D(noise, -1, kernel)
    noise = np.clip(noise, 0, 1)
    return noise * mask * strength * 0.12

# ═══ FACIAL SHEEN (CONTOUR-AWARE BROAD SPECULAR) ══════════════════════════
def _facial_sheen(arr, mask, strength=1.0):
    '''Broad wet gloss following facial contours -- not uniform shine.'''
    gray = cv2.cvtColor(arr.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    # Sheen targets mid-to-bright areas (not shadows)
    sheen = (gray - 0.25).clip(0, 1) ** 0.7
    sheen *= mask * strength * 0.14
    # Organic patchiness: low-frequency Perlin-like field
    h, w = arr.shape[:2]
    n1 = cv2.GaussianBlur(np.random.rand(h//8, w//8).astype(np.float32), (0,0), 3)
    n1 = cv2.resize(n1, (w, h))
    n2 = cv2.GaussianBlur(np.random.rand(h//4, w//4).astype(np.float32), (0,0), 2)
    n2 = cv2.resize(n2, (w, h))
    noise = n1 * 0.7 + n2 * 0.3
    sheen *= (0.35 + noise * 0.65)
    # Edge emphasis: more shine at mask edges (contour highlights)
    edge = _face_detail_mask(mask, grad_scale=4.0)
    sheen += edge * strength * 0.06
    return np.clip(sheen, 0, 1)

# ═══ WATER STREAKS / RIVULETS ═════════════════════════════════════════════
def _water_streaks(h, w, mask, strength=1.0):
    '''Gravity-driven water streaks with variable width and specular centers.'''
    spec_layer = np.zeros((h, w), dtype=np.float32)
    dark_layer = np.zeros((h, w), dtype=np.float32)
    ys, xs = np.where(mask > 0.30)
    if len(ys) < 100:
        return spec_layer, dark_layer
    y_mid = float(np.median(ys))
    for _ in range(int(12 * strength)):
        # Start from upper portion of face
        upper = np.where(ys < y_mid)[0]
        if len(upper) == 0:
            continue
        ti = upper[random.randint(0, len(upper)-1)]
        sy, sx = int(ys[ti]), int(xs[ti])
        pts = [(sx, sy)]
        # Natural flow: mostly downward with slight horizontal drift
        for _ in range(random.randint(30, 80)):
            lx, ly = pts[-1]
            ny = ly + random.randint(2, 4)
            nx = lx + random.choice([-2,-1,0,1,2])
            if 0 <= ny < h and 0 <= nx < w and mask[ny, nx] > 0.10:
                pts.append((nx, ny))
            else:
                break
        if len(pts) > 8:
            arr_pts = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)
            width = random.choice([1, 1, 2, 2, 3])
            # Dark line (water absorbing light) -- wider, softer
            cv2.polylines(dark_layer, [arr_pts], False, 0.55, width+2)
            # Specular center (bright reflection from water surface) -- thinner
            cv2.polylines(spec_layer, [arr_pts], False, 0.85, max(1, width-1))
    dark_layer = cv2.GaussianBlur(dark_layer, (5, 5), 1.2)
    spec_layer = cv2.GaussianBlur(spec_layer, (3, 3), 0.6)
    return spec_layer * strength, dark_layer * strength

# ═══ EDGE WATER ACCUMULATION ══════════════════════════════════════════════
def _edge_wetness_alpha(mask, strength=1.0):
    '''Returns (edge_dark_alpha, edge_sat_alpha) for additive compositing.'''
    edge = _face_detail_mask(mask, grad_scale=6.0)
    edge_dark = np.clip(edge * strength * 0.18, 0, 1)
    edge_sat  = np.clip(edge * strength * 0.12, 0, 1)
    return edge_dark, edge_sat

# ═══ WET SKIN TONE ════════════════════════════════════════════════════════
def _wet_tone_alpha(mask, strength=1.0):
    '''Returns (darken_alpha, sat_alpha) -- pure additive compositing only.'''
    darken = np.clip(mask * strength * 0.10, 0, 1)
    sat_boost = np.clip(mask * strength * 0.08, 0, 1)
    return darken, sat_boost

# ═══════════════════════════════════════════════════════════════════════════
# MAIN GENERATION FUNCTION
# ═══════════════════════════════════════════════════════════════════════════
def generate_wet_face(dry_img_pil, strength=STRENGTH):
    '''
    100% ADDITIVE wetness -- original face pixels NEVER changed.
    All water effects composited on top of pristine original.
    Pass 1 -- Wet skin tone (additive darken + sat via overlay/soft_light)
    Pass 2 -- Facial sheen (contour-aware specular gloss, screen blend)
    Pass 3 -- Edge wetness (additive darken + sat at facial contours)
    Pass 4 -- Water droplets (cluster-scattered caustic + fresnel, screen)
    Pass 5 -- Micro-texture (water in skin pores, soft-light blend)
    Pass 6 -- Water streaks (gravity rivulets: dark core + specular)
    '''
    img = dry_img_pil.resize((512, 512), Image.LANCZOS)
    arr = np.array(img).astype(np.float32)  # ORIGINAL -- never modified
    h, w = arr.shape[:2]
    mask = create_face_mask(img, mtcnn)
    
    # Start with pristine original as base
    result = arr.copy()
    
    # Pass 1: Wet skin tone (additive -- darken + sat as overlays)
    w_dark, w_sat = _wet_tone_alpha(mask, strength)
    dark_ovl = np.stack([w_dark]*3, axis=-1)*255
    result = _blend(result, dark_ovl*0.6, w_dark*0.7, 'soft_light')
    sat_ovl = np.stack([w_sat]*3, axis=-1)*255
    result = _blend(result, sat_ovl*1.5, w_sat*0.5, 'overlay')
    
    # Pass 2: Facial sheen (screen blend -- only brightens, never darkens)
    sheen = _facial_sheen(result, mask, strength)
    result = _blend(result, np.stack([sheen]*3, axis=-1)*255, sheen*0.85, 'screen')
    
    # Pass 3: Edge wetness (additive darken + sat at contours)
    e_dark, e_sat = _edge_wetness_alpha(mask, strength)
    e_dark_ovl = np.stack([e_dark]*3, axis=-1)*255
    result = _blend(result, e_dark_ovl, e_dark*0.65, 'soft_light')
    e_sat_ovl = np.stack([e_sat]*3, axis=-1)*255
    result = _blend(result, e_sat_ovl, e_sat*0.45, 'overlay')
    
    # Pass 4: Water droplets (screen blend on top)
    d_hl, d_al = _scatter_droplets(h, w, mask, strength)
    d_ovl = np.stack([d_hl*0.9]*3, axis=-1)*255
    result = _blend(result, d_ovl, np.clip(d_al*1.1, 0, 1), 'screen')
    
    # Pass 5: Micro-texture (soft-light blend -- subtle but realistic)
    micro = _micro_texture(h, w, mask, strength)
    micro_ovl = np.stack([micro]*3, axis=-1)*255
    result = _blend(result, micro_ovl, micro*0.8, 'soft_light')
    
    # Pass 6: Water streaks (dark core + specular center)
    streak_spec, streak_dark = _water_streaks(h, w, mask, strength)
    # Darken first (additive via soft_light)
    dark_ovl2 = np.stack([streak_dark]*3, axis=-1)*255
    result = _blend(result, dark_ovl2, streak_dark*0.7, 'soft_light')
    # Then bright specular center lines (screen = additive highlight)
    spec_ovl2 = np.stack([streak_spec]*3, axis=-1)*255
    result = _blend(result, spec_ovl2, streak_spec*0.75, 'screen')
    
    return Image.fromarray(np.clip(result, 0, 255).astype(np.uint8))

print('\n✅ Additive wetness generator ready')
print('   Mode              : 100% ADDITIVE COMPOSITING (no pixel modification)')
print(f'   Strength           : {STRENGTH}')
print('   Water droplets    : cluster-scattered caustic + fresnel (1-14px)')
print('   Micro-texture     : multi-octave noise (water in skin pores)')
print('   Facial sheen      : contour-aware specular + organic patchiness')
print('   Edge wetness      : water pooling at facial contours')
print('   Streaks/rivulets  : dark core + specular center, gravity flow')
print('   Blend modes       : screen + overlay + soft_light')
print('   Guarantee          : original face pixels NEVER touched')



## 🔍 Cell 5 -- Quality Filter Functions

In [ ]:
import cv2

def blur_score(img_pil: Image.Image) -> float:
    """Laplacian variance -- higher = sharper. Reject < 80."""
    gray = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def has_face(img_pil: Image.Image) -> bool:
    """Returns True if MTCNN detects a face. Ensures identity is preserved."""
    boxes, probs = mtcnn.detect(img_pil)
    if boxes is None or probs is None:
        return False
    return any(p > 0.90 for p in probs)  # confidence > 90%

def wetness_heuristic(img_pil: Image.Image) -> float:
    """
    Photometric wetness score -- matches the heuristic in FajrGuard's
    useWuduDetector.ts (used as fallback before TFLite model is ready).
    
    Measures specular highlights (bright spots from moisture) + 
    skin luminance reduction (wet skin appears darker/more saturated).
    Returns 0.0-1.0
    """
    img_cv = cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)
    
    # Specular ratio: % of pixels that are very bright (highlights from water)
    gray = cv2.cvtColor(img_cv, cv2.COLOR_BGR2GRAY)
    _, specular_mask = cv2.threshold(gray, 220, 255, cv2.THRESH_BINARY)
    specular_ratio = np.sum(specular_mask > 0) / specular_mask.size
    
    # Saturation: wet skin is more saturated than dry
    hsv = cv2.cvtColor(img_cv, cv2.COLOR_BGR2HSV)
    avg_saturation = np.mean(hsv[:, :, 1]) / 255.0
    
    # Combine: specular highlights matter most for wudu detection
    score = min(1.0, (specular_ratio * 8.0) + (avg_saturation * 0.3))
    return score

def is_acceptable(img_pil: Image.Image, min_blur: float = 80) -> tuple:
    """Full quality gate. Returns (passed: bool, reason: str)"""
    b = blur_score(img_pil)
    if b < min_blur:
        return False, f"blurry ({b:.0f})"
    if not has_face(img_pil):
        return False, "no face detected"
    w = wetness_heuristic(img_pil)
    if w < 0.05:  # generated image doesn't look wet at all
        return False, f"insufficient wetness ({w:.2f})"
    return True, "ok"

print("✅ Quality filter functions ready")


## 🧪 Cell 6 -- Quick Test (1 Image Before Full Run)

In [ ]:
import matplotlib.pyplot as plt

# Test on first CelebA image
test_files = sorted(os.listdir(CELEBA_DIR))[:1]
test_path  = os.path.join(CELEBA_DIR, test_files[0])

print(f"Testing on: {test_files[0]}")
dry_test = Image.open(test_path).convert("RGB")

print("Generating wet version...")
wet_test = generate_wet_face(dry_test)

passed, reason = is_acceptable(wet_test)
w_score = wetness_heuristic(wet_test)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(dry_test.resize((512, 512)))
axes[0].set_title("DRY (CelebA source)", fontsize=13)
axes[0].axis('off')
axes[1].imshow(wet_test)
axes[1].set_title(f"WET (generated)\nQuality: {reason} | Wetness score: {w_score:.2f}", fontsize=13)
axes[1].axis('off')
plt.tight_layout()
plt.show()

print(f"\nQuality check: {'✅ PASSED' if passed else '❌ FAILED'} -- {reason}")
print(f"Wetness heuristic score: {w_score:.3f} (app threshold: {WET_THRESHOLD})")
print("\nIf the wet image looks good, proceed to Cell 7 for the full pipeline.")


## 🚀 Cell 7 -- Full Automated Pipeline

In [ ]:
import csv
from tqdm import tqdm
from datetime import datetime

celeba_files = sorted([
    f for f in os.listdir(CELEBA_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])[:MAX_IMAGES]

print(f"🎯 Processing {len(celeba_files):,} images → target: {OUTPUT_DIR}")
print(f"   Strength: {STRENGTH} | Steps: {STEPS}")
print("-" * 60)

saved   = []
skipped = []
errors  = []

csv_path = f"{OUTPUT_DIR}/metadata.csv"

# Resume support -- skip already processed images
already_done = set()
if os.path.exists(csv_path):
    with open(csv_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get('status') == 'ok':
                already_done.add(row['source_file'])
    print(f"📌 Resuming -- {len(already_done)} already processed")

with open(csv_path, 'a', newline='') as csvfile:
    writer = csv.writer(csvfile)
    if len(already_done) == 0:  # write header only on fresh start
        writer.writerow(['id', 'source_file', 'dry_path', 'wet_path',
                         'wetness_score', 'blur_score', 'status', 'reason', 'timestamp'])

    for fname in tqdm(celeba_files, desc="Generating pairs"):
        if fname in already_done:
            continue

        img_id   = fname.rsplit('.', 1)[0]
        dry_path = f"{OUTPUT_DIR}/dry/{img_id}_dry.jpg"
        wet_path = f"{OUTPUT_DIR}/wet/{img_id}_wet.jpg"
        src_path = os.path.join(CELEBA_DIR, fname)
        ts       = datetime.now().isoformat()

        try:
            # 1. Load source
            dry_img = Image.open(src_path).convert("RGB")

            # 2. Pre-check source has a face
            if not has_face(dry_img):
                writer.writerow([img_id, fname, '', '', 0, 0, 'skipped', 'no face in source', ts])
                skipped.append(fname)
                continue

            # 3. Generate wet version
            wet_img = generate_wet_face(dry_img)

            # 4. Quality gate
            passed, reason = is_acceptable(wet_img)
            w_score = wetness_heuristic(wet_img)
            b_score = blur_score(wet_img)

            if not passed:
                rej_path = f"{OUTPUT_DIR}/rejected/{img_id}_wet_rejected.jpg"
                wet_img.save(rej_path, quality=75)
                writer.writerow([img_id, fname, dry_path, rej_path, f"{w_score:.3f}",
                                 f"{b_score:.1f}", 'rejected', reason, ts])
                skipped.append(fname)
                continue

            # 5. Save pair
            dry_img.resize((512, 512), Image.LANCZOS).save(dry_path, quality=92)
            wet_img.save(wet_path, quality=92)

            writer.writerow([img_id, fname, dry_path, wet_path,
                             f"{w_score:.3f}", f"{b_score:.1f}", 'ok', 'ok', ts])
            csvfile.flush()  # write immediately in case session drops
            saved.append(img_id)

        except Exception as e:
            writer.writerow([img_id, fname, '', '', 0, 0, 'error', str(e)[:80], ts])
            errors.append((fname, str(e)))

print(f"\n{'='*60}")
print(f"✅ Pairs saved:  {len(saved):,}")
print(f"⏭  Skipped:     {len(skipped):,}")
print(f"❌ Errors:       {len(errors):,}")
print(f"📄 Metadata:     {csv_path}")
print(f"{'='*60}")

# ═══ Merge real wet face benchmark images into the wet/ directory ════════
# These scraped real-world wet faces are mixed into training so the model
# learns from BOTH synthetic and real wet skin -- not just synthetic.
BENCHMARK_DIR = f'{OUTPUT_DIR}/benchmark_wet'
real_wet_added = 0
if os.path.exists(BENCHMARK_DIR):
    bench_files = sorted([f for f in os.listdir(BENCHMARK_DIR) if f.endswith('.jpg')])
    for bf in bench_files:
        src = os.path.join(BENCHMARK_DIR, bf)
        dst = os.path.join(f'{OUTPUT_DIR}/wet', f'real_{bf}')
        if not os.path.exists(dst):
            try:
                img = Image.open(src).convert('RGB')
                img = img.resize((512, 512), Image.LANCZOS)
                # Verify it has a face
                if has_face(img):
                    img.save(dst, quality=92)
                    real_wet_added += 1
            except Exception:
                pass
    # Add real wet entries to metadata.csv
    if real_wet_added > 0:
        ts = datetime.now().isoformat()
        with open(csv_path, 'a', newline='') as csvfile:
            writer = csv.writer(csvfile)
            for bf in bench_files:
                dst = os.path.join(f'{OUTPUT_DIR}/wet', f'real_{bf}')
                if os.path.exists(dst):
                    img_id = f'real_{bf.rsplit(".", 1)[0]}'
                    writer.writerow([img_id, bf, '', dst, 0, 0, 'ok', 'real_wet_benchmark', ts])
    print(f"🌊 Real wet benchmark images added to training set: {real_wet_added}")
else:
    print('⚠️  No benchmark_wet directory -- run Cell 2b first (web scraper)')
    print('   Training will use only synthetic wet faces.')
print(f"{'='*60}")


## 📊 Cell 8 -- Dataset Summary & Preview

In [ ]:
import csv
import matplotlib.pyplot as plt
import random

# Read metadata
rows = []
with open(csv_path, 'r') as f:
    reader = csv.DictReader(f)
    rows = list(reader)

ok_rows = [r for r in rows if r['status'] == 'ok']
rej_rows = [r for r in rows if r['status'] == 'rejected']

print(f"📦 Dataset Summary")
print(f"   Total processed : {len(rows):,}")
print(f"   Accepted pairs  : {len(ok_rows):,}")
print(f"   Rejected        : {len(rej_rows):,}")
print(f"   Acceptance rate : {len(ok_rows)/max(len(rows),1)*100:.1f}%")

if ok_rows:
    scores = [float(r['wetness_score']) for r in ok_rows if r.get('wetness_score', '').strip()]
    if scores:
        print(f"
   Wetness scores  : min={min(scores):.2f} | mean={sum(scores)/len(scores):.2f} | max={max(scores):.2f}")
    print(f"   App threshold   : {WET_THRESHOLD} (scores above this = confirmed wudu)")

# Preview grid -- random sample (skip real-wet entries with no dry counterpart)
sample_rows = [r for r in ok_rows if r.get('dry_path', '').strip() and r.get('wet_path', '').strip()]
sample = random.sample(sample_rows, min(3, len(sample_rows))) if sample_rows else []
if sample:
    fig, axes = plt.subplots(len(sample), 2, figsize=(10, 4 * len(sample)))
    if len(sample) == 1: axes = [axes]
    
    for i, row in enumerate(sample):
        dry = Image.open(row['dry_path'])
        wet = Image.open(row['wet_path'])
        axes[i][0].imshow(dry)
        axes[i][0].set_title(f"DRY -- {row['id']}", fontsize=11)
        axes[i][0].axis('off')
        axes[i][1].imshow(wet)
        axes[i][1].set_title(f"WET -- wetness: {row['wetness_score']}", fontsize=11)
        axes[i][1].axis('off')
    
    plt.suptitle("FajrGuard Dataset -- Random Sample", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


## Cell 8b -- Model Benchmark: Holdout Evaluation + Per-Category Analysis
Runs on the holdout real wet faces (excluded from training) + dry faces + synthetic wet faces.
Evaluates the trained model after Cell 9/10/11 are run.
Produces precision-recall curves, F1 at threshold, confidence calibration, and per-category breakdown.


In [ ]:
# ============================================================
# COMPREHENSIVE MODEL BENCHMARK
# Runs AFTER model training (Cell 9/10/11)
# Evaluates on:
#   1. Holdout real wet faces (NOT seen during training)
#   2. Holdout dry faces (CelebA not used in training)
#   3. Holdout synthetic wet faces (generated, not used in training)
# ============================================================
import torch, os, glob, random, csv, json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, classification_report,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_curve
)

# --- Define and load best model ---
import torch.nn as nn
from torchvision import models as tv_models

model = tv_models.mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 2)
)
model = model.to(DEVICE)

model_save = f"{OUTPUT_DIR}/../wudu_detector_pytorch.pt"
if os.path.exists(model_save):
    model.load_state_dict(torch.load(model_save, map_location=DEVICE))
    model.eval()
    print("Loaded trained model checkpoint")
else:
    print("=" * 65)
    print("WARNING: No trained model found. Using untrained weights.")
    print(f"Expected: {model_save}")
    print("Run Cells 9-11 first to train the model.")
    print("Benchmark results will be meaningless until model is trained.")
    print("=" * 65)

bench_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def predict_batch(img_paths, batch_size=32):
    # Run inference on a list of image paths, return probs & preds.
    probs, preds = [], []
    for i in range(0, len(img_paths), batch_size):
        batch_paths = img_paths[i:i + batch_size]
        tensors = []
        for p in batch_paths:
            try:
                img = Image.open(p).convert("RGB")
                tensors.append(bench_tf(img))
            except Exception:
                tensors.append(torch.zeros(3, 224, 224))
        if not tensors: continue
        batch = torch.stack(tensors).to(DEVICE)
        with torch.no_grad():
            out = model(batch)
            prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            pred = out.argmax(1).cpu().numpy()
        probs.extend(prob)
        preds.extend(pred)
    return np.array(probs), np.array(preds)

# --- Gather holdout sets ---
print("=" * 65)
print("FAJRGARD -- MODEL BENCHMARK (Holdout Evaluation)")
print("=" * 65)

# 1. Holdout real wet faces (from benchmark_wet/)
BENCHMARK_DIR = f"{OUTPUT_DIR}/benchmark_wet"
holdout_real_paths = sorted(glob.glob(f"{BENCHMARK_DIR}/real_wet_holdout_*.jpg"))
n_holdout_real = len(holdout_real_paths)

# 2. Holdout dry faces (CelebA images NOT used in pairs)
# Read metadata to find which CelebA images were used
csv_path = f"{OUTPUT_DIR}/metadata.csv"
used_dry = set()
used_wet = set()
if os.path.exists(csv_path):
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            if row.get("status") == "ok" and row.get("dry_path", "").strip():
                used_dry.add(os.path.basename(row["dry_path"]))
            if row.get("status") == "ok" and row.get("wet_path", "").strip():
                used_wet.add(os.path.basename(row["wet_path"]))

# Select dry faces NOT used in training
all_celeba = sorted([f for f in os.listdir(CELEBA_DIR)
                      if f.lower().endswith((".jpg", ".jpeg", ".png"))])
# Map celeb filenames to dry filenames
holdout_dry_paths = []
for cf in all_celeba:
    img_id = cf.rsplit(".", 1)[0]
    dry_name = f"{img_id}_dry.jpg"
    if dry_name not in used_dry:
        dry_path = f"{OUTPUT_DIR}/dry/{dry_name}"
        if os.path.exists(dry_path):
            holdout_dry_paths.append(dry_path)
# Limit to same count as holdout real for balance
random.shuffle(holdout_dry_paths)
holdout_dry_paths = holdout_dry_paths[:min(len(holdout_dry_paths), max(50, n_holdout_real))]

# 3. Holdout synthetic wet (generated, paired with holdout dry if available)
holdout_synth_paths = []
for dp in holdout_dry_paths:
    wp = dp.replace("_dry.jpg", "_wet.jpg").replace("/dry/", "/wet/")
    if os.path.exists(wp) and os.path.basename(wp) not in used_wet:
        holdout_synth_paths.append(wp)
holdout_synth_paths = holdout_synth_paths[:min(50, len(holdout_synth_paths))]

print(f"Holdout real wet faces    : {n_holdout_real}")
print(f"Holdout dry faces         : {len(holdout_dry_paths)}")
print(f"Holdout synthetic wet     : {len(holdout_synth_paths)}")

if n_holdout_real == 0:
    print("No holdout real wet faces found. Skipping benchmark.")
    print("Run Cell 2b first to import real-wet-faces dataset.")
else:
    # --- Run inference on all holdout sets ---
    real_probs, real_preds = predict_batch(holdout_real_paths)
    dry_probs, dry_preds = predict_batch(holdout_dry_paths)
    synth_probs, synth_preds = predict_batch(holdout_synth_paths)

    real_labels = np.ones(len(real_probs), dtype=int)
    dry_labels  = np.zeros(len(dry_probs), dtype=int)
    synth_labels = np.ones(len(synth_probs), dtype=int)

    # Combine all
    all_probs  = np.concatenate([dry_probs, synth_probs, real_probs])
    all_preds  = np.concatenate([dry_preds, synth_preds, real_preds])
    all_labels = np.concatenate([dry_labels, synth_labels, real_labels])

    # ============================================================
    # 1. OVERALL METRICS
    # ============================================================
    print("")
    print("=" * 65)
    print("1. OVERALL CLASSIFICATION REPORT (holdout)")
    print("=" * 65)
    print(classification_report(all_labels, all_preds, target_names=["Dry", "Wet"], digits=4))

    acc  = accuracy_score(all_labels, all_preds)
    auc  = roc_auc_score(all_labels, all_probs)
    f1   = f1_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec  = recall_score(all_labels, all_preds)
    ap   = average_precision_score(all_labels, all_probs)

    print(f"Accuracy          : {acc:.4f}")
    print(f"ROC-AUC           : {auc:.4f}")
    print(f"Avg Precision     : {ap:.4f}")
    print(f"F1 Score          : {f1:.4f}")
    print(f"Precision (Wet)   : {prec:.4f}")
    print(f"Recall (Wet)      : {rec:.4f}")

    # ============================================================
    # 2. PER-CATEGORY BREAKDOWN
    # ============================================================
    print("")
    print("=" * 65)
    print("2. PER-CATEGORY BREAKDOWN")
    print("=" * 65)

    def category_stats(name, probs, preds, labels):
        if len(probs) == 0: return
        acc = accuracy_score(labels, preds)
        f1c = f1_score(labels, preds, zero_division=0)
        mean_conf = np.mean(probs)
        median_conf = np.median(probs)
        correct = (preds == labels)
        mean_correct_conf = np.mean(probs[correct]) if correct.any() else 0.0
        mean_wrong_conf = np.mean(probs[~correct]) if (~correct).any() else 0.0
        print(f"  {name:25s} | N={len(probs):4d} | Acc={acc:.4f} | F1={f1c:.4f} | "
              f"Conf: mean={mean_conf:.3f} median={median_conf:.3f} "
              f"| Correct-conf={mean_correct_conf:.3f} Wrong-conf={mean_wrong_conf:.3f}")

    category_stats("Dry (CelebA holdout)", dry_probs, dry_preds, dry_labels)
    category_stats("Synthetic Wet (holdout)", synth_probs, synth_preds, synth_labels)
    category_stats("Real Wet (holdout)", real_probs, real_preds, real_labels)

    # ============================================================
    # 3. THRESHOLD ANALYSIS (F1 at app threshold 0.82)
    # ============================================================
    print("")
    print("=" * 65)
    print("3. THRESHOLD ANALYSIS")
    print("=" * 65)

    APP_THRESHOLD = 0.82
    threshold_preds = (all_probs >= APP_THRESHOLD).astype(int)
    t_acc = accuracy_score(all_labels, threshold_preds)
    t_f1  = f1_score(all_labels, threshold_preds, zero_division=0)
    t_prec = precision_score(all_labels, threshold_preds, zero_division=0)
    t_rec  = recall_score(all_labels, threshold_preds, zero_division=0)

    # Specifically on real wet faces at threshold
    real_tp = np.sum((real_probs >= APP_THRESHOLD) & (real_labels == 1))
    real_fn = np.sum((real_probs < APP_THRESHOLD) & (real_labels == 1))
    real_fp = np.sum((dry_probs >= APP_THRESHOLD) & (dry_labels == 0)) if len(dry_labels) > 0 else 0
    real_tn = np.sum((dry_probs < APP_THRESHOLD) & (dry_labels == 0)) if len(dry_labels) > 0 else 0

    print(f"At app threshold {APP_THRESHOLD}:")
    print(f"  Overall     : Acc={t_acc:.4f}  F1={t_f1:.4f}  Prec={t_prec:.4f}  Rec={t_rec:.4f}")
    print(f"  Real wet    : Detected={real_tp}/{len(real_labels)} ({real_tp/max(len(real_labels),1)*100:.1f}%)")
    print(f"  Dry false + : {real_fp}/{len(dry_labels)} ({real_fp/max(len(dry_labels),1)*100:.1f}%)")

    # Find optimal threshold (max F1)
    prec_curve, rec_curve, thresh_curve = precision_recall_curve(all_labels, all_probs)
    f1_curve = 2 * (prec_curve * rec_curve) / (prec_curve + rec_curve + 1e-10)
    best_idx = np.argmax(f1_curve)
    best_thresh = thresh_curve[best_idx] if best_idx < len(thresh_curve) else 1.0
    best_f1 = f1_curve[best_idx]
    print(f"  Optimal threshold: {best_thresh:.3f} (F1={best_f1:.4f})")

    # ============================================================
    # 4. VISUALIZATION (3x2 grid)
    # ============================================================
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # 4a. Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    import seaborn as sns
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0,0],
                xticklabels=["Dry", "Wet"], yticklabels=["Dry", "Wet"])
    axes[0,0].set_title("Confusion Matrix (Holdout)", fontweight="bold")
    axes[0,0].set_ylabel("Actual"); axes[0,0].set_xlabel("Predicted")

    # 4b. Precision-Recall Curve (better than ROC for imbalanced data)
    axes[0,1].plot(rec_curve, prec_curve, "b-", linewidth=2, label=f"AP={ap:.4f}")
    axes[0,1].plot(rec_curve[best_idx], prec_curve[best_idx], "ro",
                   markersize=8, label=f"Best F1={best_f1:.3f} @ {best_thresh:.3f}")
    axes[0,1].axhline(y=0.5, color="gray", linestyle=":", alpha=0.5)
    axes[0,1].set_xlabel("Recall"); axes[0,1].set_ylabel("Precision")
    axes[0,1].set_title("Precision-Recall Curve", fontweight="bold")
    axes[0,1].legend(loc="lower left"); axes[0,1].grid(True, alpha=0.3)

    # 4c. ROC Curve
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    axes[0,2].plot(fpr, tpr, "b-", linewidth=2, label=f"AUC={auc:.4f}")
    axes[0,2].plot([0,1], [0,1], "k--", alpha=0.3, label="Random")
    axes[0,2].set_xlabel("False Positive Rate"); axes[0,2].set_ylabel("True Positive Rate")
    axes[0,2].set_title("ROC Curve", fontweight="bold")
    axes[0,2].legend(loc="lower right"); axes[0,2].grid(True, alpha=0.3)

    # 4d. Confidence Distribution per Category
    bins = np.linspace(0, 1, 31)
    axes[1,0].hist(dry_probs, bins=bins, alpha=0.6, label="Dry (CelebA)", color="#C9A227", density=True)
    axes[1,0].hist(synth_probs, bins=bins, alpha=0.5, label="Synthetic Wet", color="#F59E0B", density=True)
    axes[1,0].hist(real_probs, bins=bins, alpha=0.5, label="Real Wet", color="#2DD4BF", density=True)
    axes[1,0].axvline(x=APP_THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"Threshold={APP_THRESHOLD}")
    axes[1,0].axvline(x=best_thresh, color="green", linestyle="--", linewidth=1.5, label=f"Optimal={best_thresh:.3f}")
    axes[1,0].set_xlabel("Predicted Wet Probability"); axes[1,0].set_ylabel("Density")
    axes[1,0].set_title("Confidence Distribution per Category", fontweight="bold")
    axes[1,0].legend(fontsize=8)

    # 4e. Calibration / Reliability Diagram
    from sklearn.calibration import calibration_curve
    prob_true, prob_pred = calibration_curve(all_labels, all_probs, n_bins=10, strategy="uniform")
    axes[1,1].plot(prob_pred, prob_true, "bo-", linewidth=2, markersize=6, label="Model")
    axes[1,1].plot([0,1], [0,1], "k--", alpha=0.3, label="Perfect calibration")
    axes[1,1].set_xlabel("Mean Predicted Probability"); axes[1,1].set_ylabel("Fraction of Positives")
    axes[1,1].set_title("Reliability Diagram (Calibration)", fontweight="bold")
    axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

    # 4f. F1 vs Threshold
    axes[1,2].plot(thresh_curve, f1_curve[:-1], "g-", linewidth=2)
    axes[1,2].axvline(x=APP_THRESHOLD, color="red", linestyle="--", label=f"App={APP_THRESHOLD}")
    axes[1,2].axvline(x=best_thresh, color="green", linestyle="--", label=f"Optimal={best_thresh:.3f}")
    axes[1,2].set_xlabel("Threshold"); axes[1,2].set_ylabel("F1 Score")
    axes[1,2].set_title("F1 Score vs Threshold", fontweight="bold")
    axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3)

    plt.suptitle("FajrGuard -- Comprehensive Model Benchmark (Holdout Data)",
                 fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ============================================================
    # 5. SUMMARY VERDICT
    # ============================================================
    print("")
    print("=" * 65)
    print("5. BENCHMARK SUMMARY")
    print("=" * 65)

    real_detect_rate = real_tp / max(len(real_labels), 1)
    dry_false_pos_rate = real_fp / max(len(dry_labels), 1)

    print(f"  Real wet detection rate : {real_detect_rate:.2%}")
    print(f"  Dry false-positive rate : {dry_false_pos_rate:.2%}")
    print(f"  AUC-ROC                 : {auc:.4f}")
    print(f"  Avg Precision           : {ap:.4f}")
    print(f"  F1 at optimal threshold : {best_f1:.4f} (thresh={best_thresh:.3f})")

    if real_detect_rate >= 0.95 and dry_false_pos_rate <= 0.03:
        print("  VERDICT: EXCELLENT -- Model meets high-accuracy requirements for wudu detection")
    elif real_detect_rate >= 0.90 and dry_false_pos_rate <= 0.05:
        print("  VERDICT: GOOD -- Model is reliable but may need more real wet training data")
    elif real_detect_rate >= 0.85 and dry_false_pos_rate <= 0.08:
        print("  VERDICT: ACCEPTABLE -- Consider increasing MAX_IMAGES or real wet faces count")
    else:
        print("  VERDICT: NEEDS IMPROVEMENT -- Increase training data (MAX_IMAGES, real wet samples)")
        print("            or tune the wetness generation parameters (STRENGTH, droplet settings)")

    # Export benchmark results
    bench_results = {
        "accuracy": float(acc), "auc_roc": float(auc), "avg_precision": float(ap),
        "f1_score": float(f1), "precision": float(prec), "recall": float(rec),
        "f1_at_app_threshold": float(t_f1),
        "optimal_threshold": float(best_thresh), "optimal_f1": float(best_f1),
        "real_wet_detection_rate": float(real_detect_rate),
        "dry_false_positive_rate": float(dry_false_pos_rate),
        "n_real_wet_holdout": n_holdout_real,
        "n_dry_holdout": len(holdout_dry_paths),
        "n_synthetic_holdout": len(holdout_synth_paths),
    }
    bench_path = f"{OUTPUT_DIR}/../benchmark_results.json"
    with open(bench_path, "w") as f:
        json.dump(bench_results, f, indent=2)
    print(f"  Benchmark results saved to: {bench_path}")



## 🧠 Cell 9 -- Train MobileNetV2 Wudu Classifier
*(Matches `ml/train/train_wudu_model.py` in FajrGuard monorepo)*

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
import os, csv, random

# ─── Dataset ─────────────────────────────────────────────────────────────────
class WuduDataset(Dataset):
    """Dry/wet face dataset. Label: 0=dry, 1=wet.
    Loads BOTH synthetic pairs AND real wet benchmark faces.
    Real wet faces (no dry pair) are still used as class-1 samples."""
    def __init__(self, metadata_csv, transform=None):
        self.samples = []
        n_synthetic_wet = 0
        n_real_wet = 0
        with open(metadata_csv) as f:
            for row in csv.DictReader(f):
                if row['status'] != 'ok': continue
                # Dry sample (from CelebA source)
                if row.get('dry_path', '').strip():
                    self.samples.append((row['dry_path'], 0))  # dry = class 0
                # Wet sample -- could be synthetic (paired) or real (unpaired)
                if row.get('wet_path', '').strip():
                    self.samples.append((row['wet_path'], 1))  # wet = class 1
                    if row.get('reason', '') == 'real_wet_benchmark':
                        n_real_wet += 1
                    else:
                        n_synthetic_wet += 1
        random.shuffle(self.samples)
        self.transform = transform
        n_dry = sum(1 for _, lbl in self.samples if lbl == 0)
        n_wet = sum(1 for _, lbl in self.samples if lbl == 1)
        print(f"Dataset: {len(self.samples)} samples")
        print(f"  Dry  : {n_dry}")
        print(f"  Wet  : {n_wet} ({n_synthetic_wet} synthetic + {n_real_wet} real)")

    def __len__(self): return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label

# Augmentations -- matches ml/train/augment.py
train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Split 80/20
full_ds = WuduDataset(csv_path, transform=train_tf)
n_val   = max(1, int(0.2 * len(full_ds)))
n_train = len(full_ds) - n_val
train_ds, val_ds = torch.utils.data.random_split(full_ds, [n_train, n_val])
val_ds.dataset.transform = val_tf

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2)

# ─── Model (MobileNetV2 -- matches wudu_detector.tflite architecture) ─────────
model = models.mobilenet_v2(weights='IMAGENET1K_V1')
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280, 2)   # 2 classes: dry / wet
)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

print(f"\n✅ MobileNetV2 ready -- {n_train} train / {n_val} val samples")


In [ ]:
# Training loop
EPOCHS    = 15
best_acc  = 0.0
model_save = f"{OUTPUT_DIR}/../wudu_detector_pytorch.pt"

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss, train_correct = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        train_correct += (out.argmax(1) == labels).sum().item()

    # Validate
    model.eval()
    val_loss, val_correct = 0.0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out = model(imgs)
            val_loss += criterion(out, labels).item() * imgs.size(0)
            val_correct += (out.argmax(1) == labels).sum().item()

    scheduler.step()

    t_acc = train_correct / n_train
    v_acc = val_correct / n_val
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train loss: {train_loss/n_train:.4f} acc: {t_acc:.3f} | "
          f"Val loss: {val_loss/n_val:.4f} acc: {v_acc:.3f}"
          + (" ← best" if v_acc > best_acc else ""))

    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), model_save)

print(f"\n✅ Training complete. Best val accuracy: {best_acc:.3f}")
print(f"   Saved: {model_save}")

## 📦 Cell 10 -- Export to TFLite (INT8 Quantized)
*(Matches `ml/export/export_tflite.py` -- produces the final `wudu_detector.tflite`)*

In [ ]:
# ═══ PyTorch → TFLite Export ═══════════════════════════════════════
# Tries tf2onnx first, then ai-edge-torch as fallback.
import subprocess, sys, os
import torch
import numpy as np
from PIL import Image

# Install onnxscript (required by torch.onnx.export in PyTorch >= 2.5)
subprocess.run([sys.executable, "-m", "pip", "install", "onnx", "onnxscript", "-q"], check=False, capture_output=True)
import onnx

model_save = f"{OUTPUT_DIR}/../wudu_detector_pytorch.pt"
tflite_path = f"{OUTPUT_DIR}/../wudu_detector.tflite"
onnx_path   = f"{OUTPUT_DIR}/../wudu_detector.onnx"

# Reload best model
model.load_state_dict(torch.load(model_save, map_location=DEVICE))
model.eval()

# --- Step 1: PyTorch -> ONNX ---
print("Step 1: PyTorch -> ONNX...")
dummy_input = torch.randn(1, 3, 224, 224).to(DEVICE)
torch.onnx.export(
    model, dummy_input, onnx_path,
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=17,
    dynamo=False
)
print(f"  ONNX: {onnx_path} ({os.path.getsize(onnx_path)/1e6:.1f} MB)")

# --- Step 2: ONNX -> TFLite (try methods) ---
# --- Step 2: ONNX -> TFLite ---
print("Step 2: ONNX -> TFLite...")
success = False

# Method A: onnx2tf (most reliable direct ONNX->TFLite)
try:
    print("  Trying onnx2tf...")
    subprocess.run([sys.executable, "-m", "pip", "install", "onnx2tf", "-q"], check=True, capture_output=True)
    import onnx2tf
    onnx2tf.convert(
        input_onnx_file_path=onnx_path,
        output_folder_path=f"{OUTPUT_DIR}/../tflite_export",
        output_signaturedefs=True,
        copy_onnx_input_output_names_to_tflite=True,
        non_verbose=True,
    )
    import glob as _g
    tflite_files = _g.glob(f"{OUTPUT_DIR}/../tflite_export/*.tflite")
    if tflite_files:
        import shutil as _sh
        _sh.copy(tflite_files[0], tflite_path)
        print(f"  onnx2tf: TFLite saved ({os.path.getsize(tflite_path)/1e6:.1f} MB)")
        success = True
except Exception as e:
    print(f"  onnx2tf failed: {type(e).__name__}: {e}")

# Method B: litert-torch (direct PyTorch -> TFLite)
if not success:
    try:
        print("  Trying litert-torch...")
        subprocess.run([sys.executable, "-m", "pip", "install", "litert-torch", "-q"], check=True, capture_output=True)
        import litert_torch
        litert_torch.convert(model.cpu().eval(), (torch.randn(1, 3, 224, 224),), tflite_path)
        print(f"  litert-torch: TFLite saved ({os.path.getsize(tflite_path)/1e6:.1f} MB)")
        success = True
    except Exception as e:
        print(f"  litert-torch failed: {type(e).__name__}: {e}")

# Method C: tf2onnx (updated API)
if not success:
    try:
        print("  Trying tf2onnx (updated API)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "tf2onnx", "-q"], check=True, capture_output=True)
        import tf2onnx
        import tensorflow as tf
        onnx_model = onnx.load(onnx_path)
        tf_rep = tf2onnx.convert.from_onnx(onnx_model)
        converter = tf.lite.TFLiteConverter.from_concrete_functions([tf_rep])
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
        tflite_model = converter.convert()
        with open(tflite_path, "wb") as f:
            f.write(tflite_model)
        print(f"  tf2onnx: TFLite saved ({os.path.getsize(tflite_path)/1e6:.1f} MB)")
        success = True
    except Exception as e:
        print(f"  tf2onnx failed: {type(e).__name__}: {e}")

if not success:
    print()
    print("=" * 65)
    print("WARNING: TFLite conversion failed.")
    print("ONNX model saved: " + onnx_path)
    print("Manual conversion options:")
    print("  1. onnx2tf -i wudu_detector.onnx")
    print("  2. Use Netron to inspect model, then convert via TF")
    print("=" * 65)
else:
    print(f"\nTFLite model: {tflite_path}")
    print(f"Size: {os.path.getsize(tflite_path)/1e6:.1f} MB")
    print("Copy this to: mobile/assets/models/wudu_detector.tflite")



## 📈 Cell 11 -- Evaluate Model (Confusion Matrix + ROC)
*(Matches `ml/evaluate/metrics.py`)*

In [ ]:
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, classification_report,
    precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json

model.load_state_dict(torch.load(model_save, map_location=DEVICE))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(DEVICE)
        out  = model(imgs)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        preds = out.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

probs_arr = np.array(all_probs)
labels_arr = np.array(all_labels)

cm   = confusion_matrix(all_labels, all_preds)
auc  = roc_auc_score(all_labels, all_probs)
ap   = average_precision_score(all_labels, all_probs)
f1   = f1_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds)
rec  = recall_score(all_labels, all_preds)

print(classification_report(all_labels, all_preds, target_names=["Dry", "Wet"], digits=4))
print(f"ROC-AUC          : {auc:.4f}")
print(f"Avg Precision    : {ap:.4f}")
print(f"F1 Score         : {f1:.4f}")
print(f"Precision (Wet)  : {prec:.4f}")
print(f"Recall (Wet)     : {rec:.4f}")

# Optimize threshold from validation set
prec_curve, rec_curve, thresh_curve = precision_recall_curve(all_labels, all_probs)
f1_curve = 2 * (prec_curve * rec_curve) / (prec_curve + rec_curve + 1e-10)
best_idx = np.argmax(f1_curve)
best_thresh = thresh_curve[best_idx] if best_idx < len(thresh_curve) else 1.0
best_f1 = f1_curve[best_idx]

# App threshold metrics
APP_THRESHOLD = float(WET_THRESHOLD)
app_preds = (probs_arr >= APP_THRESHOLD).astype(int)
app_f1 = f1_score(all_labels, app_preds, zero_division=0)
app_prec = precision_score(all_labels, app_preds, zero_division=0)
app_rec = recall_score(all_labels, app_preds, zero_division=0)

print(f"")
print(f"At app threshold {APP_THRESHOLD}: F1={app_f1:.4f}  Prec={app_prec:.4f}  Rec={app_rec:.4f}")
print(f"Optimal threshold : {best_thresh:.3f} (F1={best_f1:.4f})")

# --- Figure: 2x2 Grid ---
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Confusion Matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0,0],
            xticklabels=["Dry", "Wet"], yticklabels=["Dry", "Wet"])
axes[0,0].set_title("Confusion Matrix", fontweight="bold")
axes[0,0].set_ylabel("Actual"); axes[0,0].set_xlabel("Predicted")

# 2. Precision-Recall Curve
from sklearn.metrics import PrecisionRecallDisplay
PrecisionRecallDisplay.from_predictions(all_labels, all_probs, ax=axes[0,1])
axes[0,1].set_title(f"Precision-Recall (AP={ap:.4f})", fontweight="bold")
axes[0,1].grid(True, alpha=0.3)

# 3. Probability Distribution
axes[1,0].hist(probs_arr[labels_arr == 0], bins=35, alpha=0.6, label="Dry faces", color="#C9A227", density=True)
axes[1,0].hist(probs_arr[labels_arr == 1], bins=35, alpha=0.5, label="Wet faces", color="#2DD4BF", density=True)
axes[1,0].axvline(x=APP_THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"App={APP_THRESHOLD}")
axes[1,0].axvline(x=best_thresh, color="green", linestyle="--", linewidth=1.5, label=f"Optimal={best_thresh:.3f}")
axes[1,0].set_title("Wet Probability Distribution", fontweight="bold")
axes[1,0].set_xlabel("Predicted wet probability"); axes[1,0].set_ylabel("Density")
axes[1,0].legend(fontsize=8)

# 4. F1 vs Threshold
axes[1,1].plot(thresh_curve, f1_curve[:-1], "g-", linewidth=2)
axes[1,1].axvline(x=APP_THRESHOLD, color="red", linestyle="--", label=f"App={APP_THRESHOLD}")
axes[1,1].axvline(x=best_thresh, color="green", linestyle="--", label=f"Optimal={best_thresh:.3f}")
axes[1,1].set_xlabel("Threshold"); axes[1,1].set_ylabel("F1 Score")
axes[1,1].set_title("F1 Score vs Threshold", fontweight="bold")
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.suptitle("FajrGuard -- Model Evaluation (Validation Set)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Save metrics for deployment
eval_metrics = {
    "accuracy": float((cm[0,0] + cm[1,1]) / cm.sum()),
    "auc_roc": float(auc), "avg_precision": float(ap),
    "f1_score": float(f1), "precision": float(prec), "recall": float(rec),
    "app_threshold": APP_THRESHOLD,
    "app_threshold_f1": float(app_f1),
    "optimal_threshold": float(best_thresh), "optimal_f1": float(best_f1),
    "confusion_matrix": [[int(cm[0,0]), int(cm[0,1])], [int(cm[1,0]), int(cm[1,1])]],
}
metrics_path = f"{OUTPUT_DIR}/../eval_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(eval_metrics, f, indent=2)
print(f"Evaluation metrics saved to: {metrics_path}")


## 🔮 Cell 11b -- Test Model on New Images (Wet vs Dry)
Drop test images in a `test_images/` folder and this cell predicts
whether each face is wet or dry.


In [ ]:
# Predict wet vs dry on new test images
import os, glob
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(model_save, map_location=DEVICE))
model.eval()

test_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

TEST_DIR = "test_images"
if not os.path.exists(TEST_DIR):
    os.makedirs(TEST_DIR, exist_ok=True)
    print(f"Created '{TEST_DIR}/' folder. Add test images there and re-run.")
else:
    test_files = sorted(glob.glob(f"{TEST_DIR}/*.jpg") +
                        glob.glob(f"{TEST_DIR}/*.png") +
                        glob.glob(f"{TEST_DIR}/*.jpeg"))
    if len(test_files) == 0:
        print(f"No images in '{TEST_DIR}/'. Add .jpg/.png files and re-run.")
    else:
        print(f"Testing {len(test_files)} images...")
        print(f"{'Image':30s} | {'Pred':6s} | {'Dry %':>7s} | {'Wet %':>7s}")
        print("-" * 62)
        results = []
        for fpath in test_files:
            fname = os.path.basename(fpath)
            img = Image.open(fpath).convert("RGB")
            tensor = test_tf(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                out = model(tensor)
                probs = torch.softmax(out, dim=1)[0]
                dry_p, wet_p = probs[0].item(), probs[1].item()
                pred = "WET" if wet_p > dry_p else "DRY"
            results.append((fname, pred, dry_p, wet_p, img))
            print(f"{fname:30s} | {pred:6s} | {dry_p:6.1%} | {wet_p:6.1%}")
        # Show grid
        n = len(results)
        cols = min(4, n)
        rows = (n + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
        if rows == 1 and cols == 1:
            axes = [[axes]]
        elif rows == 1:
            axes = [axes]
        elif cols == 1:
            axes = [[ax] for ax in axes]
        for idx, (fname, pred, dry_p, wet_p, img) in enumerate(results):
            r, c = idx // cols, idx % cols
            axes[r][c].imshow(img)
            c = "#2DD4BF" if pred == "WET" else "#EF4444"
            axes[r][c].set_title(f"{pred} (wet={wet_p:.0%})", color=c, fontsize=12, fontweight="bold")
            axes[r][c].axis("off")
        for idx in range(n, rows*cols):
            r, c = idx // cols, idx % cols
            axes[r][c].axis("off")
        plt.suptitle("Wudu Detector - Test Predictions", fontsize=14, fontweight="bold")
        plt.tight_layout()
        plt.show()
        wet_n = sum(1 for _, p, _, _, _ in results if p == "WET")
        dry_n = len(results) - wet_n
        print(f"\nSummary: {wet_n} wet, {dry_n} dry out of {len(results)} images")


In [ ]:
csv_path = f"{OUTPUT_DIR}/metadata.csv"
print("FajrGuard ML Pipeline - Final Checklist")
print("=" * 50)

tflite_exists = os.path.exists(tflite_path)
if tflite_exists:
    tflite_size = os.path.getsize(tflite_path) / 1e6
    tflite_msg = f"TFLite model ({tflite_size:.1f}MB) - ready to deploy"
else:
    tflite_msg = "TFLite model (not created)"

checks = [
    (os.path.exists(csv_path),    f"metadata.csv ({len(ok_rows)} pairs)"),
    (len(os.listdir(f'{OUTPUT_DIR}/dry')) > 0,  "Dry face images saved"),
    (len(os.listdir(f'{OUTPUT_DIR}/wet')) > 0,  "Wet face images saved"),
    (os.path.exists(model_save),  "PyTorch model checkpoint (.pt)"),
    (tflite_exists, tflite_msg),
]

for passed, label in checks:
    icon = "OK" if passed else "MISSING"
    print(f"  {icon:8s} {label}")

print()
print("Next step:")
print("  cp wudu_detector.tflite fajrguard/mobile/assets/models/wudu_detector.tflite")
print()
print("The model will be loaded by useWuduDetector.ts -> react-native-fast-tflite")
print(f"Threshold in app: WUDU_THRESHOLD = {WET_THRESHOLD} (configurable in settings screen)")

